# Notebook 1: Decision Trees and Classification

**Series:** Random Forests & Isolation Forests for Full-Stack Engineers  
**Prerequisites:** Python basics, no ML/DS experience needed  
**Author:** [Farty Bobo](https://fartybobo.com)
**What you'll build:** A working decision tree classifier you actually understand

---

## The Software Engineering Analogy

Imagine you're writing a function that takes an incoming HTTP request and routes it to the right service. You look at the path, the headers, the payload size, and you make a series of `if/else` decisions:

```python
def route_request(request):
    if request.path.startswith('/api/v2'):
        if request.content_length > 1_000_000:  # > 1MB
            return 'heavy-processing-service'
        else:
            return 'api-service'
    elif request.path.startswith('/static'):
        return 'cdn'
    else:
        return 'default-service'
```

That nested `if/else` structure? **That's a decision tree.** Literally. Each `if` is a "node" that asks a yes/no question about one feature. The final `return` is a "leaf" — the answer.

A Decision Tree classifier is the same thing, except:
- You don't write the `if/else` rules by hand
- The algorithm **learns** the best rules automatically from your labeled data
- "Best" means: rules that most cleanly separate your categories

---

## What is Classification?

**Classification** = given some input features, predict which *category* (class) the input belongs to.

Examples:
- Given server metrics → is this request `normal` or `anomalous`?
- Given email text → is this `spam` or `not spam`?
- Given flower measurements → is this flower `setosa`, `versicolor`, or `virginica`?

We'll use that last one. It's the "Hello World" of ML classification.

---

## The Iris Dataset

The [Iris flower dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set) has **150 flower samples** described by 4 numeric measurements:

| Feature | Description |
|---|---|
| `sepal length (cm)` | Length of the sepal (outer petal) |
| `sepal width (cm)` | Width of the sepal |
| `petal length (cm)` | Length of the petal |
| `petal width (cm)` | Width of the petal |

Each sample is labeled as one of 3 species: `setosa`, `versicolor`, or `virginica`.

Think of it like a database table where each row is an observation and we know the "answer" (the label) for every row. This **labeled data** is what we use to train the model.

> **Why Iris?** It's clean, small, and has some features that perfectly separate classes and some that don't — great for learning how classifiers work.

In [ ]:
# Import the standard scientific Python stack.
# Think of these like npm packages for data/viz work.
import numpy as np               # numerical arrays (like typed buffers)
import matplotlib.pyplot as plt  # plotting
import seaborn as sns            # nicer-looking plots built on matplotlib
import pandas as pd              # tabular data (like a spreadsheet in memory)

# sklearn = scikit-learn, the standard ML library for Python
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Set up a colorblind-friendly palette
pal = sns.color_palette('colorblind')
sns.set_palette(pal)

print('All imports successful!')

In [ ]:
# Load the Iris dataset. 
# sklearn bundles several classic datasets — load_iris() is like importing a fixture.
iris = load_iris()

# iris.data is a 2D numpy array: 150 rows (samples), 4 columns (features)
# iris.target is a 1D array of integers: 0=setosa, 1=versicolor, 2=virginica
# iris.feature_names tells us what each column is
# iris.target_names tells us what each integer label means

print(f'Data shape: {iris.data.shape}  →  {iris.data.shape[0]} samples, {iris.data.shape[1]} features')
print(f'Features:   {iris.feature_names}')
print(f'Classes:    {iris.target_names}')
print()

# Let's look at the first 5 rows as a DataFrame (think: database query result)
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = [iris.target_names[t] for t in iris.target]
df.head()

### Visualizing the Data

Before building any model, **always look at your data**. This is the equivalent of reading your API docs or schema before writing code.

We'll plot pairs of features against each other, color-coded by species. If you can visually separate the colored dots, a classifier can probably do it too.

In [ ]:
# A pairplot shows every feature plotted against every other feature.
# Each dot is one iris sample. Color = species.
# The diagonal shows the distribution of each single feature.
sns.pairplot(df, hue='species', diag_kind='hist', height=2.2)
plt.suptitle('Iris Dataset: Every Feature vs Every Other Feature', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**What to notice:**
- The orange `setosa` cluster is completely separated from the other two in nearly every plot involving petal measurements.
- `versicolor` and `virginica` overlap somewhat — they're harder to separate.
- `petal length` and `petal width` seem like the most useful features for classification.

This gives us intuition for what the decision tree will "learn".

---

## Training a Decision Tree

In sklearn, training a model is always the same pattern:

```python
model = SomeClassifier(hyperparameters...)
model.fit(X, y)   # X = features array, y = labels array
model.predict(X_new)  # make predictions
```

This is like: instantiate a service → configure it → call it.

The `fit()` call is where the learning happens. It builds the if/else tree from your labeled data.

In [ ]:
# Train a decision tree on the full iris dataset.
# random_state=0 means: fix the random seed so results are reproducible.
# (Like seeding a test database — same seed = same data every run.)

clf = DecisionTreeClassifier(random_state=0)
clf.fit(iris.data, iris.target)  # iris.data = X (features), iris.target = y (labels)

print(f'Tree depth: {clf.get_depth()}')
print(f'Number of leaves (final decision nodes): {clf.get_n_leaves()}')
print(f'Training accuracy: {clf.score(iris.data, iris.target):.1%}')
# Note: training accuracy on the data we trained on isn't a fair measure!
# We'll cover train/test splits in the next notebook.

In [ ]:
# Visualize the actual if/else tree the algorithm learned.
# This is the full decision logic — you can trace any sample through it manually.

fig, ax = plt.subplots(figsize=(16, 10), dpi=100)
plot_tree(
    clf,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,        # color boxes by majority class
    rounded=True,
    ax=ax
)
plt.title('The Decision Tree — Your Learned if/else Logic', fontsize=14)
plt.tight_layout()
plt.show()

### Reading the Tree

Each box (node) shows:
- **The condition** (e.g., `petal width (cm) <= 0.8`) — this is the `if` statement
- **gini** — how "mixed" the samples at this node are (we'll explain this next)
- **samples** — how many training samples reached this node
- **value** — the count per class `[setosa, versicolor, virginica]`
- **class** — the majority class at this node

**Trace an example:** A flower with `petal width = 0.5 cm`:
1. Root node: `petal width <= 0.8` → TRUE → go left
2. Left leaf: 50 setosa, 0 versicolor, 0 virginica → **predict: setosa** ✓

That's it. One question, done. `setosa` is perfectly separable from the others by petal width.

---

## The Gini Impurity: How the Tree Decides Where to Split

The tree has to decide: **which feature to split on, and at what value?**

It uses the **Gini Impurity** to score candidate splits. Think of it as measuring "how mixed is this bucket?"

- **Gini = 0.0** → perfectly pure, all one class. Like a list that contains only `setosa`.
- **Gini = 0.5** → maximum impurity for 2 classes. Like a 50/50 coin flip — zero information.

The formula for two classes:

$$G = p(1-p) + (1-p)(1-(1-p)) = 2p(1-p)$$

Where $p$ is the fraction of class 0 in the node.

The tree tries millions of candidate splits and picks the one that **minimizes** Gini impurity (maximizes purity) in the resulting child nodes.

> **Analogy:** Imagine sorting a pile of mixed playing cards into two piles. A "good split" is one where pile A has mostly hearts and pile B has mostly spades — low Gini on both sides. A "bad split" randomly divides the pile with no improvement — high Gini on both sides.

In [ ]:
# Let's visualize Gini impurity, entropy, and misclassification error
# to understand what these metrics actually measure.

# These are three different ways to measure "impurity" of a node.
# The x-axis is p = fraction of class 1 in the node.

def gini(p):
    """Gini impurity for 2-class case. Max at p=0.5 (equal mix), min at p=0 or p=1 (pure)."""
    return 2 * p * (1 - p)

def entropy(p):
    """Information-theoretic entropy. Penalizes uncertainty more than Gini."""
    if p == 0 or p == 1:
        return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def misclassification_error(p):
    """Simple error rate: what fraction would you get wrong if you always predicted the majority class?"""
    return 1 - max(p, 1 - p)

p_values = np.linspace(0.001, 0.999, 500)  # range of class fractions

gini_vals  = [gini(p) for p in p_values]
entropy_vals = [entropy(p) for p in p_values]
error_vals = [misclassification_error(p) for p in p_values]

fig, ax = plt.subplots(figsize=(9, 5), dpi=100)

ax.plot(p_values, gini_vals,    label='Gini Impurity',          lw=2.5, color=pal.as_hex()[0])
ax.plot(p_values, [e/2 for e in entropy_vals], label='Entropy (scaled to 0.5)', lw=2.5, color=pal.as_hex()[1], linestyle='--')
ax.plot(p_values, error_vals,   label='Misclassification Error', lw=2.5, color=pal.as_hex()[2], linestyle='-.')

ax.axhline(y=0.5, lw=1, color='gray', linestyle=':')
ax.axvline(x=0.5, lw=1, color='gray', linestyle=':', label='p=0.5 (maximum uncertainty)')

ax.set_xlabel('p  (fraction of class 1 in node)', fontsize=12)
ax.set_ylabel('Impurity', fontsize=12)
ax.set_title('How Impurity Measures Score a Node', fontsize=13)
ax.legend(loc='upper center', fontsize=10)
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the Chart

All three metrics share the same shape:
- At `p=0` or `p=1` (all one class → **pure node**): impurity = **0**. Perfect.
- At `p=0.5` (50/50 mix → **maximally confused**): impurity peaks.

The tree's job is to find splits that push nodes toward `p=0` or `p=1`.

**Why Gini over Entropy?** Gini is cheaper to compute (no logarithm) and behaves similarly in practice. sklearn defaults to Gini.

---

## How the Decision Tree Algorithm Works (Step by Step)

Here's the full algorithm in pseudocode — this is exactly what sklearn's `DecisionTreeClassifier.fit()` does:

```
function build_tree(data, labels):
    # Base cases — stop splitting
    if all labels are the same:    return leaf(class=that_label)
    if data has 0 rows:            return leaf(class=majority_class)
    if max_depth reached:          return leaf(class=majority_class)

    best_split = None
    best_gini  = infinity

    # Try every feature
    for feature in random_subset(all_features):
        # Try many possible threshold values for this feature
        for threshold in sample_thresholds(data[feature]):
            left  = data[data[feature] <= threshold]
            right = data[data[feature] >  threshold]
            
            # Weighted average Gini of the two resulting buckets
            gini = weighted_gini(left, right)
            
            if gini < best_gini:
                best_gini  = gini
                best_split = (feature, threshold)

    # Recurse on each half
    left_data,  right_data  = split(data, best_split)
    return node(
        condition  = best_split,
        left_child = build_tree(left_data,  ...),
        right_child= build_tree(right_data, ...)
    )
```

It's a greedy recursive algorithm — at each step it picks the locally best split, without backtracking. Sound familiar? It's a lot like a greedy search.

---

## Making Predictions

In [ ]:
# Let's manually create some hypothetical flowers and classify them.
# The features are: [sepal_length, sepal_width, petal_length, petal_width]

hypothetical_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],   # very small petals -> probably setosa
    [6.0, 2.9, 4.5, 1.5],   # medium petals -> probably versicolor
    [6.9, 3.1, 5.5, 2.2],   # large petals -> probably virginica
])

# predict() returns the integer class label
predictions = clf.predict(hypothetical_flowers)
# predict_proba() returns the probability for each class — more useful!
probabilities = clf.predict_proba(hypothetical_flowers)

print('Predictions:')
for i, (pred, probs) in enumerate(zip(predictions, probabilities)):
    class_name = iris.target_names[pred]
    prob_str = ', '.join(f'{iris.target_names[j]}: {p:.0%}' for j, p in enumerate(probs))
    print(f'  Flower {i+1}: {class_name:12s}  (probs: {prob_str})')

## The Problem: Overfitting

Wait — we got 100% accuracy on the training data! Isn't that perfect?

**No. It's a red flag.** This is called **overfitting**.

The tree we built was allowed to grow as deep as it wanted. It memorized every single training example — including random noise and quirks in the data. It's like a student who memorized every practice test answer verbatim instead of understanding the subject. They ace practice tests but fail on new questions.

The real test of a model is how it performs on **data it hasn't seen before**. We'll cover this and how to prevent overfitting with proper **train/test splits** and **hyperparameter tuning** in the next notebooks.

---

## Summary

| Concept | What it is | Software analogy |
|---|---|---|
| **Classification** | Predicting which category an input belongs to | Request routing / type narrowing |
| **Decision Tree** | A learned series of if/else rules on features | Your hand-written router, but auto-generated from data |
| **Gini Impurity** | Measures how "mixed" a node's classes are | How unsorted a bucket is |
| **Training** | Fitting the model to labeled data via `fit(X, y)` | Building an index from your data |
| **Overfitting** | Model memorized training data, fails on new data | Hardcoded test values instead of real logic |

---

## What's Next

**Notebook 2: Feature Preparation** — Your raw data is almost never in the right shape for a decision tree. We'll learn why feature distributions matter and how to transform them.

In [ ]:
# === CHECK YOUR UNDERSTANDING ===
# Try modifying the code below and re-running to build intuition.

# 1. What happens when you limit max_depth to 2?
#    The tree becomes simpler. Does accuracy go down?
clf_shallow = DecisionTreeClassifier(max_depth=2, random_state=0)
clf_shallow.fit(iris.data, iris.target)
print(f'Shallow tree (max_depth=2) training accuracy: {clf_shallow.score(iris.data, iris.target):.1%}')

fig, ax = plt.subplots(figsize=(12, 6), dpi=100)
plot_tree(clf_shallow, feature_names=iris.feature_names, class_names=iris.target_names, filled=True, rounded=True, ax=ax)
plt.title('Shallow Decision Tree (max_depth=2)', fontsize=13)
plt.tight_layout()
plt.show()

# 2. Questions to think about:
# - Which feature did the tree use first (root node)? Why is that the best first split?
# - The shallow tree has lower accuracy. Is that always bad? (Hint: think about new data)
# - What does 'gini = 0.0' in a leaf node mean?